In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


###Lang Smith 토큰 발급하기###
https://smith.langchain.com/

In [ ]:
# 설치
!pip install -U langchain langchain_openai

In [ ]:
import os

In [ ]:
with open("/content/drive/MyDrive/AI/인사교_LangChain_20260624/key/lang_smith_key.txt", "r") as f:
    api_key = f.read().strip()

LANGSMITH_TRACING="true"
LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
LANGSMITH_API_KEY=api_key
LANGSMITH_PROJECT="aischool0701"

os.environ['LANGSMITH_TRACING'] = "true"
os.environ['LANGSMITH_ENDPOINT'] = "https://api.smith.langchain.com"
os.environ['LANGSMITH_API_KEY'] = api_key
os.environ['LANGSMITH_PROJECT'] = "aischool0701"

In [ ]:
# OpenAI Key 환경변수 설정
with open("/content/drive/MyDrive/AI/인사교_LangChain_20260624/key/openai_key.txt", "r") as f:
    api_key = f.read().strip()

os.environ['OPENAI_API_KEY'] = api_key

In [ ]:
#기본 체인 만들어서 호출하기
from langchain.chat_models import init_chat_model
# 프롬프트
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# 아웃풋 파서
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# 모델생성
llm_4o_mini = init_chat_model("openai:gpt-4o-mini", max_tokens=1024)
# 프롬프트 템플릿 생성
template = ChatPromptTemplate.from_messages([
    ("system","아래 사용자 입력 글을 스페인어로 번역해줘. 출력형식은 다른 멘트없이 번역글만 나오게해"),
    ("human","userInput : {input}")
])
#문자열 파서
strParser = StrOutputParser()

In [ ]:
# 체인 구성하기
translate_chain = template | llm_4o_mini | strParser

In [ ]:
# 체인 호출하기
print(translate_chain.invoke({"input" : " 노란 티셔츠를 입은 아이는 너의 사촌이야?"}))

¿El niño que lleva la camiseta amarilla es tu primo?


###Lang Serve 활용하기
- 직접 구성한 chain을 API형태로 호출 할 수 있게 Lang serve로 구축
- ngrok을 활용해 코랩의 GPU환경을 MVP용으로 사용할 수 있도록 구축
- 실제 프로덕션 단계에서는 직접 서버 환경을 구축할 필요가 있으나 MVP시연용으로는 가능

### 패키지 설치하기
-기본 : langchain, langchain-openai => 체인구성을 위한 기본 패키지
- 추가
 - langserve: 구성한 체인은 REST API로 감싸는 패키지
 - fastapi, uvicorn: 파이썬 서버를 구성하기 위한 도구
 - pyngrok : 터널링을 이용해 외부통신을 가능하게 해주는 도구

In [ ]:
!pip install langserve fastapi uvicorn pyngrok

In [ ]:
!pip install sse_starlette

### FastAPI + LangServe 구성하기

In [ ]:
from fastapi import FastAPI
from langserve import add_routes

In [ ]:
app = FastAPI() # 서버생성
add_routes(app, # 라우팅을 추가할 서버
          translate_chain, # 라우팅에 연결할 체인
          path="/query") # 연결된 체인을 호출할 때 활용하는 URL

###Colab에서 서버 실행

In [ ]:
import nest_asyncio # 노트북 커널과 별개 수행이 가능하도록 도와주는 패키지
import uvicorn # 서버를 구동하는 패키지
from threading import Thread # 기존 노트북 커널과 별도로 쓰레드가 분할되도록 하는 도구

In [ ]:
nest_asyncio.apply()

def run() :
  uvicorn.run(app, # 구동할 서버 앱
              host='127.0.0.1', # 현재 할당받은 컴퓨팅 자원의 IP를 지칭
              port=8000) # 사용할 포트번호

thread = Thread(target=run) # 쓰레드 생성
thread.start() # 쓰레드 구동

INFO:     Started server process [1397]
INFO:     Waiting for application startup.


### Colab 인스턴스 내부에서 호출하기

In [ ]:
!curl -X POST "http://127.0.0.1:8000/query/invoke" \
-H "Content-Type: application/json" \
-d '{"input" : {"input" : "노란 티셔츠를 입은 아이는 너의 사촌이야?"}}'

INFO:     127.0.0.1:54826 - "POST /query/invoke HTTP/1.1" 200 OK
{"output":"¿El niño que lleva la camiseta amarilla es tu primo?","metadata":{"run_id":"42fef877-c35d-43dd-9e30-3635f499c9ae","feedback_tokens":[]}}

###ngrok을 활용한 외부에서 호출

In [ ]:
cd /content/drive/MyDrive/AI/인사교_LangChain_20260624/key

/content/drive/MyDrive/AI/인사교_LangChain_20260624/key


In [ ]:
import os
with open("/content/drive/MyDrive/AI/인사교_LangChain_20260624/key/ngrok_key.txt", "r") as f:
   api_key = f.read().strip()
os.system(f'ngrok config add-authtoken {api_key}')

0

In [ ]:
from pyngrok import ngrok

In [ ]:
public_url = ngrok.connect(8000) # 현재 PC에서 서버가 돌고 있는 포트 연결

In [ ]:
public_url

<NgrokTunnel: "https://crestless-dealing-anteater.ngrok-free.dev" -> "http://localhost:8000">

In [ ]:
# ngrok 연결해제
# ngrok.disconnect(public_url)

In [ ]:
!curl -X POST "https://crestless-dealing-anteater.ngrok-free.dev.app:8000/query/invoke" \
-H "Content-Type: application/json" \
-d '{"input" : {"input" : "노란 티셔츠를 입은 아이는 너의 사촌이야?"}}'

curl: (6) Could not resolve host: crestless-dealing-anteater.ngrok-free.dev.app


### POST Man으로 테스트하기
https://www.postman.com